# 02 — Train Baseline Model (A3)
SIH26182 — Duo A (Data & ML)

**Goal:** A working Random Forest classifier with real, honest metrics.


In [5]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

merged_df = pd.read_csv("merged_data.csv")
print(merged_df.shape)
merged_df.head()


(46564, 168)


,txId,time_step,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_157,feat_158,feat_159,feat_160,feat_161,feat_162,feat_163,feat_164,feat_165,label
0,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,licit
1,232029206,1,-0.005027,0.578941,-0.091383,4.380281,-0.063725,4.667146,0.851305,-0.163645,...,-0.613614,0.241128,0.241406,0.604120,0.008632,-0.131155,0.333211,-0.120613,-0.119792,licit
2,232344069,1,-0.147852,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.137933,...,-0.613614,0.241128,0.241406,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,licit
3,27553029,1,-0.151357,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.141519,...,-0.582077,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,licit
4,3881097,1,-0.172306,-0.184668,-1.201369,0.028105,-0.043875,-0.029140,0.242712,-0.163640,...,-0.600999,0.241128,0.241406,0.018279,-0.068266,-0.084674,-0.054450,-1.760926,-1.760984,licit


## Temporal train/test split (time_step 1-34 train, 35-49 test)
The Elliptic dataset has 49 time steps. A **random** split mixes transactions from all time
steps into both train and test, which lets the model implicitly see patterns from
time-adjacent transactions and inflates the score. The standard benchmark (and the honest
version for your demo) is a **temporal split**: train on earlier time steps, test on later
ones the model has never seen — same setup used in the original Elliptic paper.

In [6]:
feature_columns = [c for c in merged_df.columns if c.startswith("feat_") or c == "time_step"]

train_mask = merged_df["time_step"] <= 34
test_mask = merged_df["time_step"] > 34

X_train = merged_df.loc[train_mask, feature_columns]
y_train = merged_df.loc[train_mask, "label"]
X_test = merged_df.loc[test_mask, feature_columns]
y_test = merged_df.loc[test_mask, "label"]

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train label balance:")
print(y_train.value_counts(normalize=True))
print("Test label balance:")
print(y_test.value_counts(normalize=True))


Train shape: (29894, 166)  Test shape: (16670, 166)
Train label balance:
label
licit      0.884191
illicit    0.115809
Name: proportion, dtype: float64
Test label balance:
label
licit      0.935033
illicit    0.064967
Name: proportion, dtype: float64


## Train RandomForestClassifier (class_weight='balanced')

In [7]:
clf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

precision = precision_score(y_test, y_pred, pos_label="illicit")
recall = recall_score(y_test, y_pred, pos_label="illicit")
f1 = f1_score(y_test, y_pred, pos_label="illicit")

print(f"Precision (illicit): {precision:.3f}")
print(f"Recall (illicit):    {recall:.3f}")
print(f"F1-score (illicit):  {f1:.3f}")
print()
print("Confusion matrix (rows=true, cols=pred), labels=[illicit, licit]:")
print(confusion_matrix(y_test, y_pred, labels=["illicit", "licit"]))
print()
print(classification_report(y_test, y_pred))


Precision (illicit): 0.916
Recall (illicit):    0.725
F1-score (illicit):  0.809

Confusion matrix (rows=true, cols=pred), labels=[illicit, licit]:
[[  785   298]
 [   72 15515]]

              precision    recall  f1-score   support

     illicit       0.92      0.72      0.81      1083
       licit       0.98      1.00      0.99     15587

    accuracy                           0.98     16670
   macro avg       0.95      0.86      0.90     16670
weighted avg       0.98      0.98      0.98     16670



✅ **Verify (A3):** Real numbers, not NaN/zero. With the temporal split, F1 anywhere in **0.6–0.85 is a legitimate, honest baseline** (this matches the range reported for Random Forest in the original Elliptic paper) — GNN-level 0.9+ results come from published graph-neural-net research, not a plain Random Forest, so don't chase that number tonight. If your F1 comes out much higher than this range, double check `train_mask`/`test_mask` are actually being used (not an old `X_train`/`X_test` from a random split still sitting in memory).

## Save the trained model

In [8]:
joblib.dump(clf, "model.pkl")
print("Saved model.pkl")


Saved model.pkl


---
**CHECKPOINT 1 → share with Duo B now:**
- `model.pkl`
- A screenshot of the confusion matrix / metrics printed above
